In [6]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import re

In [10]:
hands = pl.read_parquet("../data/processed/hands.parquet")
players = pl.read_parquet("../data/processed/player_hands.parquet")
actions = pl.read_parquet("../data/processed/actions.parquet")

In [11]:
players

game_id,player,seat,stack_start,is_button,is_small_blind,is_big_blind,hole_card_1,hole_card_2,cards_known,preflop_actions,flop_actions,turn_actions,river_actions,vpip,preflop_raise,saw_flop,saw_turn,saw_river,showdown,total_bet,total_collect,net_result,won_hand
i64,str,i64,f64,i64,i64,i64,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,i64
787027613,"""StephCurry""",1,105.78,0,0,0,null,null,0,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0
787027613,"""PANDAisEVIL""",2,101.0,0,0,0,null,null,0,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0
787027613,"""AironVega""",3,103.2,0,0,0,null,null,0,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0
787027613,"""IlxxxlI""",4,43.0,0,0,0,"""9d""","""7d""",1,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0
787027613,"""pineapplesand""",5,40.0,0,0,0,null,null,0,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
751157922,"""IlxxxlI""",5,53.33,0,0,0,"""2d""","""5h""",1,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0
751158089,"""superman51210""",1,43.77,0,0,0,null,null,0,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0
751158089,"""IampokerKidd""",3,63.72,1,0,0,null,null,0,"""raise""","""bet|call""","""bet""",null,1,1,1,1,0,0,17.5,0.0,-17.5,0


In [13]:
hands.columns

['game_id',
 'start_time',
 'end_time',
 'table_name',
 'small_blind',
 'big_blind',
 'game_type',
 'button_seat',
 'num_players',
 'board_flop',
 'board_turn',
 'board_river',
 'pot_total',
 'rake',
 'jp_fee',
 'has_showdown',
 'winner_count']

In [14]:
players.columns

['game_id',
 'player',
 'seat',
 'stack_start',
 'is_button',
 'is_small_blind',
 'is_big_blind',
 'hole_card_1',
 'hole_card_2',
 'cards_known',
 'preflop_actions',
 'flop_actions',
 'turn_actions',
 'river_actions',
 'vpip',
 'preflop_raise',
 'saw_flop',
 'saw_turn',
 'saw_river',
 'showdown',
 'total_bet',
 'total_collect',
 'net_result',
 'won_hand']

In [15]:
actions.columns

['game_id', 'player', 'street', 'action_order', 'action_type', 'amount']

In [16]:
# ============================================================================
# AJOUT DES FEATURES
# ============================================================================

def calculate_position(seat, button_seat, num_players):
    """
    Calcule la position du joueur basée sur son siège et la position du button.
    """
    if seat is None or button_seat is None:
        return "UNKNOWN"

    # Calculer la position relative au button
    relative_position = (seat - button_seat) % num_players

    # Mapper les positions en fonction du nombre de joueurs
    if num_players <= 2:
        position_map = {
            0: "BTN",
            1: "BB",
        }
    elif num_players == 3:
        position_map = {
            0: "BTN",
            1: "SB",
            2: "BB",
        }
    elif num_players <= 6:  # 6-max
        position_map = {
            0: "BTN",
            1: "SB",
            2: "BB",
            3: "UTG",
            4: "MP",
            5: "CO",
        }
    else:  # Full ring
        position_map = {
            0: "BTN",
            1: "SB",
            2: "BB",
            3: "UTG",
            4: "UTG+1",
            5: "MP",
            6: "MP+1",
            7: "CO",
            8: "HJ",
        }

    if relative_position >= len(position_map):
        return f"UTG-{relative_position - 3}"

    return position_map.get(relative_position, "UNKNOWN")


In [32]:
# Joindre les données pour ajouter les features
print("Ajout de la feature 'position'...")
players_enriched_raw = players.join(
    hands.select(["game_id", "button_seat", "num_players"]),
    on="game_id",
    how="left"
)

# Calculer la position
players_enriched = players_enriched_raw.with_columns(
    pl.struct(["seat", "button_seat", "num_players"])
    .map_elements(
        lambda x: calculate_position(x["seat"], x["button_seat"], x["num_players"]),
        return_dtype=pl.Utf8
    )
    .alias("position")
)

# Réorganiser les colonnes pour avoir position après seat
final_columns = [
    "game_id", "player", "seat", "position",
    "stack_start", "is_button", "is_small_blind", "is_big_blind",
    "hole_card_1", "hole_card_2", "cards_known",
    "preflop_actions", "flop_actions", "turn_actions", "river_actions",
    "vpip", "preflop_raise", "saw_flop", "saw_turn", "saw_river",
    "showdown", "total_bet", "total_collect", "net_result", "won_hand"
]

players_enriched = players_enriched.select(final_columns)

print(f"[OK] {len(players_enriched)} joueurs enrichis")
print("\nColonnes finales:")
print(players_enriched.columns)

Ajout de la feature 'position'...
✅ 299988 joueurs enrichis

Colonnes finales:
['game_id', 'player', 'seat', 'position', 'stack_start', 'is_button', 'is_small_blind', 'is_big_blind', 'hole_card_1', 'hole_card_2', 'cards_known', 'preflop_actions', 'flop_actions', 'turn_actions', 'river_actions', 'vpip', 'preflop_raise', 'saw_flop', 'saw_turn', 'saw_river', 'showdown', 'total_bet', 'total_collect', 'net_result', 'won_hand']


In [33]:
# Afficher un aperçu des données enrichies
print("\nAperçu des joueurs avec les nouvelles features:")
players_enriched_raw


📊 Aperçu des joueurs avec les nouvelles features:


game_id,player,seat,stack_start,is_button,is_small_blind,is_big_blind,hole_card_1,hole_card_2,cards_known,preflop_actions,flop_actions,turn_actions,river_actions,vpip,preflop_raise,saw_flop,saw_turn,saw_river,showdown,total_bet,total_collect,net_result,won_hand,button_seat,num_players
i64,str,i64,f64,i64,i64,i64,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,i64,i64,i64
787027613,"""StephCurry""",1,105.78,0,0,0,null,null,0,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0,7,9
787027613,"""PANDAisEVIL""",2,101.0,0,0,0,null,null,0,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0,7,9
787027613,"""AironVega""",3,103.2,0,0,0,null,null,0,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0,7,9
787027613,"""IlxxxlI""",4,43.0,0,0,0,"""9d""","""7d""",1,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0,7,9
787027613,"""pineapplesand""",5,40.0,0,0,0,null,null,0,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0,7,9
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
751157922,"""IlxxxlI""",5,53.33,0,0,0,"""2d""","""5h""",1,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0,1,4
751158089,"""superman51210""",1,43.77,0,0,0,null,null,0,"""fold""",null,null,null,0,0,0,0,0,0,0.0,0.0,0.0,0,3,4
751158089,"""IampokerKidd""",3,63.72,1,0,0,null,null,0,"""raise""","""bet|call""","""bet""",null,1,1,1,1,0,0,17.5,0.0,-17.5,0,3,4


In [25]:
# Distribution des positions
print("\nDistribution des positions:")
position_counts = players_enriched.select("position").group_by("position").len().sort("len", descending=True)
print(position_counts)


🎲 Distribution des positions:
shape: (10, 2)
┌──────────┬───────┐
│ position ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ BTN      ┆ 58964 │
│ BB       ┆ 49517 │
│ SB       ┆ 49386 │
│ UTG      ┆ 47426 │
│ MP       ┆ 43671 │
│ CO       ┆ 29074 │
│ MP+1     ┆ 9474  │
│ UTG+1    ┆ 9200  │
│ HJ       ┆ 3211  │
│ UNKNOWN  ┆ 65    │
└──────────┴───────┘


In [26]:
# Statistiques par position
print("\nStatistiques par position:")
stats_by_position = players_enriched.group_by("position").agg([
    pl.col("vpip").mean().alias("vpip_avg"),
    pl.col("preflop_raise").mean().alias("preflop_raise_avg"),
    pl.col("net_result").mean().alias("avg_win_loss"),
    pl.col("won_hand").mean().alias("win_rate"),
    pl.len().alias("count")
]).sort("count", descending=True)

print(stats_by_position)


📈 Statistiques par position:
shape: (10, 6)
┌──────────┬──────────┬───────────────────┬──────────────┬──────────┬───────┐
│ position ┆ vpip_avg ┆ preflop_raise_avg ┆ avg_win_loss ┆ win_rate ┆ count │
│ ---      ┆ ---      ┆ ---               ┆ ---          ┆ ---      ┆ ---   │
│ str      ┆ f64      ┆ f64               ┆ f64          ┆ f64      ┆ u32   │
╞══════════╪══════════╪═══════════════════╪══════════════╪══════════╪═══════╡
│ BTN      ┆ 0.290364 ┆ 0.22183           ┆ -1.180368    ┆ 0.023811 ┆ 58964 │
│ BB       ┆ 0.239655 ┆ 0.079084          ┆ -1.637848    ┆ 0.032958 ┆ 49517 │
│ SB       ┆ 0.258778 ┆ 0.146013          ┆ -1.450412    ┆ 0.025554 ┆ 49386 │
│ UTG      ┆ 0.194008 ┆ 0.148189          ┆ -0.8285      ┆ 0.01788  ┆ 47426 │
│ MP       ┆ 0.202972 ┆ 0.165258          ┆ -0.741723    ┆ 0.016189 ┆ 43671 │
│ CO       ┆ 0.227832 ┆ 0.177203          ┆ -0.904903    ┆ 0.020362 ┆ 29074 │
│ MP+1     ┆ 0.185877 ┆ 0.128351          ┆ -0.997923    ┆ 0.017416 ┆ 9474  │
│ UTG+1    ┆ 0.1647